# 01 - Bronze Daily Updates

**Purpose:** Automated daily market data ingestion from Alpha Vantage API

**Author:** Jonah A.  
**Created:** 2025-08-04

**Architecture Layer:** Bronze (Raw Data - Daily Updates)

**Input:** Alpha Vantage TIME_SERIES_DAILY API  
**Output:** Appends to `bronze_market_data_persistent` Delta table

**Business Value:** Real-time portfolio monitoring with daily market data updates

**Schedule:** Daily at 6:30 PM ET (30 minutes after market close)  
**API Usage:** 15 requests/day (15 AI stocks × 1 request each)  
**Rate Limit:** 5 calls/minute with 12-second delays between stocks

In [0]:
# =============================================================================
# CELL #1: IMPORTS AND DAILY CONFIGURATION
# PURPOSE: Minimal setup for daily Alpha Vantage automation
# =============================================================================

from datetime import datetime, timedelta, date
import time
import requests
from pyspark.sql import Row

# Alpha Vantage API Configuration (minimal - no duplication)
ALPHA_VANTAGE_API_KEY = dbutils.secrets.get(scope="market-risk", key="alpha_vantage")
ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"

# Daily date logic (unique to daily automation)
today = date.today()
target_date = (today - timedelta(days=1)).strftime("%Y-%m-%d")

# Handle weekends
target_weekday = (today - timedelta(days=1)).weekday()
if target_weekday == 5:  # Saturday -> Friday
    target_date = (today - timedelta(days=2)).strftime("%Y-%m-%d")
elif target_weekday == 6:  # Sunday -> Friday  
    target_date = (today - timedelta(days=3)).strftime("%Y-%m-%d")

print(f"🤖 Daily Alpha Vantage Automation")
print(f"📅 Target Date: {target_date}")

In [0]:
# =============================================================================
# CELL #2: ALPHA VANTAGE FETCH FUNCTION
# PURPOSE: Alpha Vantage specific data fetching (minimal function)
# =============================================================================

def fetch_alpha_vantage_daily(symbol, target_date):
    """Fetch single day data from Alpha Vantage"""
    
    params = {
        'function': 'TIME_SERIES_DAILY',
        'symbol': symbol,
        'apikey': ALPHA_VANTAGE_API_KEY,
        'outputsize': 'compact'
    }
    
    try:
        response = requests.get(ALPHA_VANTAGE_BASE_URL, params=params)
        
        if response.status_code == 200:
            data = response.json()
            
            if "Note" in data or "Error Message" in data:
                print(f"❌ API error for {symbol}")
                return None
                
            time_series = data.get('Time Series (Daily)', {})
            
            if target_date in time_series:
                daily_data = time_series[target_date]
                print(f"✅ {symbol}: ${float(daily_data['4. close']):.2f}")
                return {
                    'symbol': symbol,
                    'date': target_date,
                    'open': float(daily_data['1. open']),
                    'high': float(daily_data['2. high']),
                    'low': float(daily_data['3. low']),
                    'close': float(daily_data['4. close']),
                    'volume': int(daily_data['5. volume'])
                }
            else:
                print(f"❌ No data for {symbol} on {target_date}")
                return None
        else:
            print(f"❌ HTTP error for {symbol}")
            return None
            
    except Exception as e:
        print(f"❌ Error fetching {symbol}: {e}")
        return None

print("🔧 Alpha Vantage function ready")

In [0]:
# =============================================================================
# CELL #3: GET PORTFOLIO AND SCHEMA FROM BRONZE (SERVERLESS COMPATIBLE)
# PURPOSE: Use Bronze layer configuration (no RDD operations)
# =============================================================================

# Get portfolio symbols from Bronze persistent table (DataFrame operations only)
portfolio_df = spark.sql("""
    SELECT DISTINCT symbol 
    FROM bronze_market_data_persistent 
    WHERE data_source = 'yahoo_finance'
    ORDER BY symbol
""")

# Convert to Python list using DataFrame collect() (no RDD)
portfolio_symbols = [row['symbol'] for row in portfolio_df.collect()]

print(f"📊 Portfolio from Bronze: {len(portfolio_symbols)} stocks")
print(f"Symbols: {portfolio_symbols}")

# Get Bronze schema by reading existing table structure
bronze_table_schema = spark.table("bronze_market_data_persistent").schema

print("🏗️ Using Bronze layer schema")

In [0]:
# =============================================================================
# CELL #4: DAILY DATA COLLECTION
# PURPOSE: Collect Alpha Vantage data with rate limiting
# =============================================================================

print(f"🚀 Starting daily collection for {target_date}")

successful_data = []
failed_symbols = []
ingestion_time = datetime.now()

for i, symbol in enumerate(portfolio_symbols, 1):
    print(f"[{i:2d}/{len(portfolio_symbols)}] {symbol}...")
    
    alpha_data = fetch_alpha_vantage_daily(symbol, target_date)
    
    if alpha_data:
        successful_data.append(alpha_data)
    else:
        failed_symbols.append(symbol)
    
    # Rate limiting: 12 seconds between calls
    if i < len(portfolio_symbols):
        time.sleep(12)

print(f"\n📊 Results: {len(successful_data)}/{len(portfolio_symbols)} successful")

In [0]:
# =============================================================================
# CELL #5: VALIDATION AND RETRY LOGIC
# PURPOSE: Validate collection completeness and retry failed stocks
# =============================================================================

expected_stocks = 15
actual_successful = len(successful_data)

if actual_successful < expected_stocks:
   print(f"⚠️  Missing {expected_stocks - actual_successful} stocks for {target_date}")
   print(f"📋 Failed stocks: {failed_symbols}")
   
   # Simple retry for failed stocks
   print("🔄 Attempting retry for failed stocks...")
   retry_data = []
   
   for symbol in failed_symbols:
       print(f"🔄 Retrying {symbol}...")
       time.sleep(15)  # Respect rate limits
       retry_result = fetch_alpha_vantage_daily(symbol, target_date)
       if retry_result:
           retry_data.append(retry_result)
           print(f"✅ Retry successful for {symbol}")
   
   # Add retry successes to main data
   successful_data.extend(retry_data)
   print(f"📊 Final result: {len(successful_data)}/{expected_stocks} stocks")
else:
   print(f"✅ Complete collection: {actual_successful}/{expected_stocks} stocks")

In [0]:
# =============================================================================
# CELL #6: SAVE TO BRONZE LAYER (WITH DUPLICATE PREVENTION)
# PURPOSE: Append to existing Bronze table with duplicate checking
# =============================================================================

if successful_data:
    print(f"\n💾 Checking for existing data for {target_date}...")
    
    # Check if Alpha Vantage data for this date already exists
    existing_count = spark.sql(f"""
        SELECT COUNT(*) as count 
        FROM bronze_market_data_persistent 
        WHERE data_source = 'alpha_vantage' AND date = '{target_date}'
    """).collect()[0]['count']
    
    if existing_count > 0:
        print(f"⚠️ Alpha Vantage data for {target_date} already exists: {existing_count} rows")
        print("🔄 Skipping ingestion to prevent duplicates")
        
        # Still show current totals
        alpha_count = spark.sql("""
            SELECT COUNT(*) as count 
            FROM bronze_market_data_persistent 
            WHERE data_source = 'alpha_vantage'
        """).collect()[0]['count']
        
        print(f"📈 Total Alpha Vantage records: {alpha_count}")
        
    else:
        print(f"📥 New data for {target_date} - proceeding with ingestion...")
        
        # Create Bronze rows using same schema as existing table
        alpha_bronze_rows = []
        for stock_data in successful_data:
            bronze_row = Row(
                ingestion_timestamp=ingestion_time,
                symbol=stock_data['symbol'],
                date=stock_data['date'],
                open=stock_data['open'],
                high=stock_data['high'],
                low=stock_data['low'],
                close=stock_data['close'],
                volume=stock_data['volume'],
                data_source="alpha_vantage"
            )
            alpha_bronze_rows.append(bronze_row)
        
        # Create DataFrame and append
        daily_df = spark.createDataFrame(alpha_bronze_rows, bronze_table_schema)
        daily_df.write.mode("append").saveAsTable("bronze_market_data_persistent")
        
        print(f"✅ Saved {len(successful_data)} rows to Bronze")
        
        # Validation
        alpha_count = spark.sql("""
            SELECT COUNT(*) as count 
            FROM bronze_market_data_persistent 
            WHERE data_source = 'alpha_vantage'
        """).collect()[0]['count']
        
        print(f"📈 Total Alpha Vantage records: {alpha_count}")
    
else:
    print("❌ No data to save")

print(f"✅ Daily automation complete for {target_date}")